# Comparison and analysis (Parts c, d, e, f)

Reads the saved results of `01_configA_cnn.ipynb` and `02_resnet_B_Bp_C.ipynb` and turns them into the
tables and figures for the report. **No training and no GPU**: only the CSVs in `outputs/` and the images
in `datasets/val` and `datasets/test` are used.

| Section | Report part |
|---|---|
| 0. Setup | — |
| 1. Load and check the results | — |
| 2. Main results table | d, f |
| 3. Comparison chain B → B′ → C-noaug → C, and A vs C | f |
| 4. Training curves and the augmentation effect | b, f |
| 5. Learning-rate sweep | c |
| 6. Error analysis of Config C | e |
| 7. Config A vs Config C, image by image | e, f |
| 8. Summary for the report | — |

**How to run:** select the `ie4483-venv` kernel, then Kernel → Restart, Run All. Every figure is also saved
to `outputs/`, so the report can use them after the cell outputs are cleared.

## 0. Setup

**Needed:**
- Imports: pandas, numpy, matplotlib, PIL, scikit-learn metrics
- Paths and constants from `common.py`: `OUTPUT_DIR`, `DATA_DIR`, `CLASS_TO_IDX`, `SEED`
- One fixed colour per config, used in every figure
- Small helpers: save a figure to `outputs/`, compute a 95% confidence interval for an accuracy

In [ ]:
from math import sqrt

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix

from common import CLASS_TO_IDX, DATA_DIR, OUTPUT_DIR, SEED

IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}   # 0 -> "cat", 1 -> "dog"

# One fixed colour per config, the same in every figure (colour-blind-safe palette).
CONFIGS = {
    "A":       {"label": "A: custom CNN (scratch)",       "color": "#2a78d6"},
    "B":       {"label": "B: frozen ResNet-18 + SVM",     "color": "#eb6834"},
    "Bp":      {"label": "B′: frozen ResNet-18 + linear", "color": "#1baf7a"},
    "C-noaug": {"label": "C-noaug: fine-tuned without aug",   "color": "#eda100"},
    "C":       {"label": "C: fine-tuned + aug (final)",   "color": "#6250d6"},
}
SWEEP = {"sweep-lr1e-3": 1e-3, "sweep-lr1e-4": 1e-4, "sweep-lr1e-5": 1e-5}
SWEEP_COLORS = {"sweep-lr1e-3": "#2a78d6", "sweep-lr1e-4": "#eb6834", "sweep-lr1e-5": "#1baf7a"}

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})


def save(fig, name):
    """Save a figure to outputs/ so the report can use it after outputs are cleared."""
    path = OUTPUT_DIR / name
    fig.savefig(path)
    print("saved", path.relative_to(OUTPUT_DIR.parent).as_posix())


def wilson_ci(acc, n, z=1.96):
    """95% confidence interval for an accuracy measured on n images (Wilson score interval)."""
    centre = (acc + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * sqrt(acc * (1 - acc) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half


print("data:", DATA_DIR, "| outputs:", OUTPUT_DIR)

## 1. Load and check the results

**Inputs:**
- `outputs/results_<config>_seed42.csv` for A, B, Bp, C-noaug, C (one row per epoch; B has one row)
- `outputs/results_sweep-lr{1e-3,1e-4,1e-5}_seed42.csv` (3 epochs each, on the subset)
- `outputs/preds_A_seed42.csv`, `outputs/preds_C_seed42.csv` (per-image val predictions)

**Checks:** every run is complete (15 epochs, sweep 3 epochs), all evaluated on `val`, all on the same GPU,
5,000 val images. Print the `git_commit` of each run (note any `-dirty` tag for the report).

In [ ]:
def load_results(config):
    return pd.read_csv(OUTPUT_DIR / f"results_{config}_seed{SEED}.csv")


results = {c: load_results(c) for c in CONFIGS}   # one DataFrame per config, one row per epoch
sweep = {c: load_results(c) for c in SWEEP}
preds = {c: pd.read_csv(OUTPUT_DIR / f"preds_{c}_seed{SEED}.csv") for c in ["A", "C"]}

N_VAL = len(preds["C"])
all_runs = pd.concat([*results.values(), *sweep.values()])

# One line per run: how many epochs, which split, GPU and code version produced it.
check = all_runs.groupby("config", sort=False).agg(
    epochs=("epoch", "max"), split=("eval_split", "first"),
    gpu=("gpu", "first"), git_commit=("git_commit", "first"))
print(check.to_string())

# Every trained config must have had the same budget, and the sweep runs the same length.
trained = ["A", "Bp", "C-noaug", "C"]
assert len({len(results[c]) for c in trained}) == 1, "trained configs ran different numbers of epochs"
assert len(results["B"]) == 1, "B (SVM) should have a single row"
assert len({len(s) for s in sweep.values()}) == 1, "sweep runs have different lengths"

# Comparable runs: all evaluated on val, all on the same GPU.
assert (all_runs.eval_split == "val").all()
assert all_runs.gpu.nunique() == 1, "runs come from different GPUs: not comparable"

# Predictions cover the whole val set, the same images for A and C.
n_val_images = sum(1 for _ in (DATA_DIR / "val").glob("*/*.jpg"))
assert len(preds["A"]) == N_VAL == n_val_images, "predictions do not cover the full val set"
assert set(preds["A"].path) == set(preds["C"].path)

dirty = sorted(all_runs.loc[all_runs.git_commit.str.endswith("-dirty"), "config"].unique())
print(f"\nAll checks passed: {len(results)} configs + {len(sweep)} sweep runs on {all_runs.gpu.iloc[0]}, "
      f"trained configs {len(results['C'])} epochs each; {N_VAL} val images (1 image = {100 / N_VAL:.2f} percentage points)")
if dirty:
    print("Note for the report: run with uncommitted changes (-dirty):", ", ".join(dirty))

## 2. Main results table (Parts d, f)

**Needed:** one row per config with
- best validation accuracy and the epoch it occurred (the checkpoint that was kept)
- 95% confidence interval and number of misclassified val images
- training accuracy and validation loss at that epoch
- starting learning rate and total training time

**Outputs:** `outputs/comparison_table.csv`, and a bar chart of misclassified val images per config
(`outputs/fig_misclassified.png`).

In [ ]:
rows = []
for c, df in results.items():
    best = df.loc[df.eval_acc.idxmax()]          # the epoch whose checkpoint was kept
    lo, hi = wilson_ci(best.eval_acc, N_VAL)
    rows.append({
        "config": c,
        "model": CONFIGS[c]["label"],
        "best_val_acc": best.eval_acc,
        "ci95_low": round(lo, 4),
        "ci95_high": round(hi, 4),
        "val_misclassified": round((1 - best.eval_acc) * N_VAL),
        "best_epoch": int(best.epoch),
        "epochs_run": int(df.epoch.max()),
        "train_acc_at_best": best.train_acc,
        "val_loss_at_best": best.eval_loss,
        "lr_start": df.lr.iloc[0],
        "train_seconds": round(df.seconds.sum(), 1),
    })
table = pd.DataFrame(rows).set_index("config")
table.to_csv(OUTPUT_DIR / "comparison_table.csv")
print("saved outputs/comparison_table.csv")
table.drop(columns="model")

In [ ]:
# Number of misclassified val images per config: fewer is better.
order = list(CONFIGS)
wrong_per_config = table.loc[order, "val_misclassified"]

fig, ax = plt.subplots(figsize=(8, 3.4))
bars = ax.barh([CONFIGS[c]["label"] for c in order], wrong_per_config,
               color=[CONFIGS[c]["color"] for c in order], height=0.6)
for bar, c in zip(bars, order):
    r = table.loc[c]
    ax.annotate(f"{int(r.val_misclassified)} misclassified  ({100 * r.best_val_acc:.2f}% accuracy)",
                (bar.get_width(), bar.get_y() + bar.get_height() / 2),
                xytext=(5, 0), textcoords="offset points", va="center", fontsize=9)
ax.invert_yaxis()
ax.set_xlim(0, wrong_per_config.max() * 1.45)   # room for the labels
ax.set_xlabel(f"Misclassified validation images (out of {N_VAL:,}), fewer is better")
ax.grid(axis="y", visible=False)

resnet = ["B", "Bp", "C-noaug", "C"]
spread = int(table.loc[resnet, "val_misclassified"].max() - table.loc[resnet, "val_misclassified"].min())
ax.set_title(f"Misclassified validation images per config\n(the four ResNet-18 configs are within {spread} images "
             f"of each other)", fontsize=11)
save(fig, "fig_misclassified.png")
plt.show()

## 3. Comparison chain (Part f)

Each step changes **exactly one thing**, so any difference can be attributed to that change:

```
B ──────────► B′ ──────────────► C-noaug ──────────► C
   SVM → FC       frozen → fine-tuned     no aug → aug
 (classifier)       (fine-tuning)         (augmentation)
```

plus **A vs C** (scratch CNN vs pretrained ResNet-18 at the same budget).

The ResNet configs differ by only a few images, so before interpreting a difference we ask **whether it is
bigger than chance**. Three checks, all from the results CSVs:

- **Noise floor:** how much a config's own val accuracy moves over its last 5 epochs, with nothing changed.
  A step whose difference is no bigger than this cannot be told apart from epoch-to-epoch wobble.
- **Mean of the last 5 epochs:** a fairer score than the best epoch, which picks the luckiest of 15 values.
  (B has a single score, so it gets no such advantage.)
- **95% interval of the difference:** combines the uncertainty of both accuracies (5,000 val images each).
  If it contains 0, we cannot say the change helped or hurt.

For augmentation (C-noaug → C) we also compare the **train–val gap** (overfitting), as the plan requires.

In [ ]:
LAST_K = 5   # epochs used for the "mean of last epochs" and the noise floor
mean_col = f"mean_last{LAST_K}_val_acc"

# Add three columns to the section 2 table (one row per config), then save it again.
for c, df in results.items():
    last = df.eval_acc.tail(LAST_K)
    single = len(df) == 1                       # B: one score, no epochs
    table.loc[c, mean_col] = table.loc[c, "best_val_acc"] if single else round(last.mean(), 5)
    table.loc[c, "noise_floor_images"] = np.nan if single else round((last.max() - last.min()) * N_VAL)
    table.loc[c, "train_val_gap_pp"] = round(100 * (table.loc[c, "train_acc_at_best"] - table.loc[c, "best_val_acc"]), 3)

table.to_csv(OUTPUT_DIR / "comparison_table.csv")
print("updated outputs/comparison_table.csv with", mean_col, "noise_floor_images, train_val_gap_pp")
print(f"noise floor = max − min val accuracy over the last {LAST_K} epochs, in images "
      f"(1 image = {100 / N_VAL:.2f} pp); B has a single score")
table[["best_val_acc", mean_col, "noise_floor_images", "train_val_gap_pp", "val_loss_at_best"]].round(4)

In [ ]:
Z95 = 1.96   # 95% two-sided

steps = [
    ("B", "Bp", "classifier: SVM → linear layer"),
    ("Bp", "C-noaug", "fine-tuning: frozen → layer4 trained"),
    ("C-noaug", "C", "augmentation: off → on"),
    ("A", "C", "scratch CNN → pretrained ResNet-18"),
]
chain = []
for a, b, what in steps:
    pa, pb = table.loc[a, "best_val_acc"], table.loc[b, "best_val_acc"]
    diff = pb - pa
    half = Z95 * sqrt(pa * (1 - pa) / N_VAL + pb * (1 - pb) / N_VAL)   # interval of a difference of two accuracies
    lo, hi = diff - half, diff + half
    verdict = "real improvement" if lo > 0 else "real decline" if hi < 0 else "within noise"
    mean_diff = table.loc[b, mean_col] - table.loc[a, mean_col]
    chain.append({
        "step": f"{a} → {b}",
        "what changes": what,
        "best acc before → after": f"{100 * pa:.2f}% → {100 * pb:.2f}%",
        "diff_pp": round(100 * diff, 2),
        "diff_ci95_pp": f"{100 * lo:+.2f} to {100 * hi:+.2f}",
        "diff_images": round(diff * N_VAL),
        f"diff_mean_last{LAST_K}_pp": round(100 * mean_diff, 2),
        "same direction": bool(np.sign(mean_diff) == np.sign(diff)),
        "noise_floor_images": table.loc[[a, b], "noise_floor_images"].max(),
        "verdict": verdict,
    })
chain = pd.DataFrame(chain)
chain.to_csv(OUTPUT_DIR / "comparison_chain.csv", index=False)
print("saved outputs/comparison_chain.csv")
print("noise_floor_images = the larger noise floor of the two configs in the step")
chain

In [ ]:
# Augmentation step in detail: train–val gap at the best epoch and over the last epochs.
aug = pd.DataFrame({c: {
    "best val acc": results[c].eval_acc.max(),
    "train acc at best epoch": results[c].loc[results[c].eval_acc.idxmax(), "train_acc"],
    "train–val gap at best epoch (pp)": table.loc[c, "train_val_gap_pp"],
    f"mean train–val gap, last {LAST_K} epochs (pp)":
        100 * (results[c].train_acc - results[c].eval_acc).tail(LAST_K).mean(),
    "lowest val loss": results[c].eval_loss.min(),
    "epoch of lowest val loss": int(results[c].loc[results[c].eval_loss.idxmin(), "epoch"]),
    "final val loss": results[c].eval_loss.iloc[-1],
    f"noise floor, last {LAST_K} epochs (images)": table.loc[c, "noise_floor_images"],
} for c in ["C-noaug", "C"]})
aug.round(5)   # 5 decimals: C-noaug's train acc is 0.99995, not 1

### What the chain shows

Read together with the tables above (the numbers are computed there):

- **B → B′ (classifier type, Week 6):** the SVM and the linear layer see the *same* 512 frozen features, and
  their difference is within noise. Once the pretrained features separate cats from dogs this well, the choice
  of classifier on top barely matters: the features do the work. B′ is even favoured by the comparison
  (best of 15 epochs vs B's single score) and still does not beat B.
- **B′ → C-noaug (fine-tuning, Week 8):** letting layer4 adapt gives a small gain, but it is within the
  interval and about the size of the noise floor, so with one seed and 5,000 images it cannot be confirmed.
  ImageNet already contains many cat and dog breeds, so frozen features leave little room to improve.
- **C-noaug → C (augmentation, Week 11):** val accuracy is the same within noise. The effect of augmentation
  shows instead in **overfitting and stability**: C-noaug fits the training set almost perfectly (largest
  train–val gap) and its val accuracy wobbles more between epochs; C has a smaller gap and a lower noise floor.
  *Caveat:* C's training accuracy is measured on augmented images, which are harder, so part of its smaller gap
  comes from that; the val-loss curves in section 4 give a cleaner view.
- **A vs C (pretraining + architecture, Weeks 8, 9):** by far the largest effect, a real improvement on every
  measure (best epoch, mean of last epochs, interval far from 0). *Limitation:* architecture and pretraining
  both differ, so this is their combined effect, not pretraining alone.

**Overall:** the choice that matters is **starting from a pretrained network**. After that, classifier type,
fine-tuning and augmentation each change only a handful of images, no more than a single model's own
epoch-to-epoch variation. C is kept as the final model because it is tied for the best accuracy and is the most
stable and least overfit of the top configs, not because it is proven more accurate.

*Limits of this comparison (state in the report):* one seed per config, so the intervals cover the choice of
validation images but not training randomness; per-image predictions exist only for A and C, so the chain steps
use unpaired intervals (A vs C is tested image by image in section 7).

## 4. Training curves and the augmentation effect (Parts b, f)

**Needed:**
- Loss and accuracy per epoch, training and validation, for A, B′, C-noaug and C (B has no curve),
  with markers where StepLR halves the learning rate → `outputs/fig_curves.png`
- C-noaug vs C: training vs validation loss and the gap between them, to show whether augmentation
  reduces overfitting → `outputs/fig_aug_gap.png`

Caveat to state: training metrics for A and C are measured on augmented images, so they are not directly
comparable with validation metrics.

## 5. Learning-rate sweep (Part c)

**Needed:** table and plot of validation loss and accuracy per epoch for lr = 1e-3, 1e-4, 1e-5
→ `outputs/fig_lr_sweep.png`, then the reason 1e-4 was chosen.

Caveat to state: the sweep ran 3 epochs on the subset (500 val images, 1 image = 0.2 percentage points).

## 6. Error analysis of Config C (Part e)

The test set has no labels, so correct/incorrect samples come from the **validation set**.

**Needed:**
- Confusion matrix and per-class precision/recall → `outputs/fig_confusion.png`
- Confidence of correct vs wrong predictions
- Examples of correct predictions → `outputs/fig_samples_correct.png`
- Examples of wrong predictions → `outputs/fig_samples_wrong.png`
- 1–2 cases chosen for discussion (strength and weakness of the model)
- A few test images with their predicted labels from `submission.csv` → `outputs/fig_test_samples.png`

## 7. Config A vs Config C, image by image (Parts e, f)

**Needed:**
- Count images C gets right but A gets wrong, and the reverse, plus a test of whether the difference
  could be chance (McNemar)
- Examples of images A gets wrong and C gets right → `outputs/fig_A_vs_C.png`

## 8. Summary for the report

**Needed:** the key numbers in one place (final accuracy with interval, chain differences, augmentation
effect, A vs C) and a list of the files saved to `outputs/`.